# 05 - Generate Load Balancer Statistics

This notebook aggregates historical data from the database and generates the `per_task_model_stats.json` file required by the Load Balancer's `StatsRegistry`.

**Output Path**: `artemis_final/ares/aggregates/per_task_model_stats.json`

In [1]:
import sys
from pathlib import Path
import json
import logging
import pandas as pd
from sqlalchemy import text


In [2]:

# Path Setup
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent
if str(ARTEMIS_DIR) not in sys.path:
    sys.path.append(str(ARTEMIS_DIR))

from ares.db.connection import get_engine
from load_balancer.config import STATS_PATH

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)-15s | %(message)s')
logger = logging.getLogger("stats_gen")

In [3]:
# Connect to Database
engine = get_engine()
print(f"Connected to DB. Stats will be saved to: {STATS_PATH}")

Connected to DB. Stats will be saved to: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/ares/aggregates/per_task_model_stats.json


In [4]:
# Query Data
query = """
SELECT 
    s.router_task,
    r.model_name,
    r.latency_ms,
    r.is_correct,
    r.total_tokens,
    r.estimated_cost_usd
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
WHERE r.ok = true 
  AND s.router_task IS NOT NULL 
  AND r.model_name IS NOT NULL
"""

print("Loading data...")
df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows.")
display(df.head())

Loading data...
Loaded 339056 rows.


,router_task,model_name,latency_ms,is_correct,total_tokens,estimated_cost_usd
0,meme_classification,deepseek_ocr,85.0,False,30,0.000002
1,meme_classification,qwen2_5_vl_3b,3488.0,True,177,0.000018
2,meme_classification,qwen2_5_vl_7b,1362.0,True,177,0.000035
3,meme_classification,qwen3_vl_8b_thinking,6359.0,True,263,0.000318
4,spatial_reasoning,deepseek_ocr,129.0,False,87,0.000004


In [5]:
# Aggregate Statistics
stats = {}

if not df.empty:
    tasks = df['router_task'].unique()
    print(f"Processing {len(tasks)} tasks: {tasks}")

    for task in tasks:
        task_df = df[df['router_task'] == task]
        stats[task] = {}
        
        models = task_df['model_name'].unique()
        for model in models:
            m_df = task_df[task_df['model_name'] == model]
            
            # Calculate metrics
            avg_latency = m_df['latency_ms'].mean()
            
            # Accuracy: handle potential nulls if filtering logic changes, though JOIN enforces sample existence
            # Explicitly checking for notna() is safer
            acc_df = m_df[m_df['is_correct'].notna()]
            avg_accuracy = acc_df['is_correct'].mean() if not acc_df.empty else 0.0
            
            # Cost per request (updated for final design)
            cost_per_request = m_df['estimated_cost_usd'].mean()
            
            stats[task][model] = {
                "avg_latency_ms": float(avg_latency),
                "avg_accuracy": float(avg_accuracy),
                "cost_per_request_usd": float(cost_per_request)
            }
else:
    print("Warning: DataFrame is empty!")

# Display sample
import json
print(json.dumps(stats, indent=2)[:500] + "...")

Processing 30 tasks: ['meme_classification' 'spatial_reasoning' 'handwriting_ocr'
 'diagram_reasoning' 'table_reasoning' 'table_math' 'document_ocr'
 'general_vqa' 'chart_reasoning' 'map_reasoning' 'dense_captioning'
 'icon_reasoning' 'chart_captioning' 'diagram_captioning' 'knowledge_vqa'
 'medical_report' 'geometry_reasoning' 'code_generation' 'ui_captioning'
 'science_reasoning' 'scene_text_ocr' 'abstract_reasoning'
 'rendered_text_ocr' 'counting' 'difference_detection' 'image_captioning'
 'textbook_qa' 'visual_mrc' 'medical_vqa' 'web_understanding']
{
  "meme_classification": {
    "deepseek_ocr": {
      "avg_latency_ms": 259.673,
      "avg_accuracy": 0.483,
      "cost_per_request_usd": 5.906015e-06
    },
    "qwen2_5_vl_3b": {
      "avg_latency_ms": 1234.0385,
      "avg_accuracy": 0.729,
      "cost_per_request_usd": 4.52544e-05
    },
    "qwen2_5_vl_7b": {
      "avg_latency_ms": 1412.8905,
      "avg_accuracy": 0.7915,
      "cost_per_request_usd": 9.05088e-05
    },
    

In [6]:
# Save to JSON
STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
    
print(f"\u2705 Statistics saved to {STATS_PATH}")

✅ Statistics saved to /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/ares/aggregates/per_task_model_stats.json
